# Silver — Transactions (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.transactions` |
| **Target** | `{catalog}.silver.transactions` |
| **SCD Type** | SCD1 — financial records update in place |
| **Depends on** | Bronze Transactions + Silver Orders (RI check) |

**DQ checks applied:**
- Cast `quantity` to INT, `unit_price` and `total_amount` to DOUBLE
- NULL or empty `payment_mode` → replaced with `Unknown`
- Flag rows where `total_amount != quantity * unit_price` with `_amount_flag = AMOUNT_MISMATCH`
- Quarantine rows with negative `total_amount` — excluded from Silver

## Setup — Widgets & Constants

Widget values are overridden at runtime by Workflow job parameters.

In [ ]:
dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')
SOURCE_TABLE  = f'{CATALOG}.{SOURCE_SCHEMA}.transactions'
TABLE         = f'{CATALOG}.{TARGET_SCHEMA}.transactions'
SILVER_ORDERS = f'{CATALOG}.{TARGET_SCHEMA}.orders'

print(f'Source          : {SOURCE_TABLE}')
print(f'Target          : {TABLE}')
print(f'RI check against: {SILVER_ORDERS}')

## Step 1 — Read from Bronze

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)
print(f'Bronze rows: {bronze_df.count()}')
bronze_df.display()

## Step 2 — DQ Checks

All numeric columns arrive as STRING from Bronze. Cast them before any arithmetic.
Flag mismatches rather than dropping — keep the row visible in Silver with a reason column.

In [ ]:
from pyspark.sql.functions import col, when, lit, to_date
from pyspark.sql.functions import round as spark_round

# Cast numeric columns + dates
casted = bronze_df \
    .withColumn('quantity',     col('quantity').cast('int')) \
    .withColumn('unit_price',   col('unit_price').cast('double')) \
    .withColumn('total_amount', col('total_amount').cast('double')) \
    .withColumn('created_at',   to_date(col('created_at'))) \
    .drop('_ingested_at', '_source_file', '_batch_id')

# NULL or empty payment_mode → 'Unknown'
casted = casted.withColumn('payment_mode',
    when(col('payment_mode').isNull() | (col('payment_mode') == ''), lit('Unknown'))
    .otherwise(col('payment_mode')))

# Flag rows where total_amount does not match quantity * unit_price
casted = casted.withColumn('_amount_flag',
    when(spark_round(col('quantity') * col('unit_price'), 2) != col('total_amount'),
         lit('AMOUNT_MISMATCH'))
    .otherwise(lit(None)))

print(f'Amount mismatches: {casted.filter(col("_amount_flag").isNotNull()).count()}')
casted.filter(col('_amount_flag').isNotNull()) \
      .select('transaction_id', 'quantity', 'unit_price', 'total_amount', '_amount_flag').display()

In [ ]:
# Quarantine negative total_amount — these are invalid financial records
valid_txn  = casted.filter(col('total_amount') >= 0)
quarantine = casted.filter(col('total_amount') < 0)

print(f'Valid transactions  : {valid_txn.count()}')
print(f'Quarantined (negative total_amount): {quarantine.count()}')
if quarantine.count() > 0:
    quarantine.select('transaction_id', 'order_id', 'total_amount').display()

In [ ]:
# Referential integrity: transaction must link to a known order in silver.orders
silver_orders = spark.table(SILVER_ORDERS).select('order_id').distinct()

ri_check = valid_txn.join(
    silver_orders.withColumnRenamed('order_id', '_order_ref'),
    valid_txn.order_id == col('_order_ref'),
    'left'
).withColumn('_ri_flag',
    when(col('_order_ref').isNull(), lit('UNKNOWN_ORDER')).otherwise(lit(None))
).drop('_order_ref')

violations = ri_check.filter(col('_ri_flag').isNotNull())
print(f'RI violations (unknown order_id): {violations.count()}')
if violations.count() > 0:
    violations.select('transaction_id', 'order_id', '_ri_flag').display()

silver_df = ri_check.filter(col('_ri_flag').isNull()).drop('_ri_flag')
print(f'Valid transactions for Silver: {silver_df.count()}')

## Step 3 — Create Silver Table (first run only)

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}')

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE} (
        transaction_id STRING,
        order_id       STRING,
        quantity       INT,
        unit_price     DOUBLE,
        total_amount   DOUBLE,
        payment_mode   STRING,
        created_at     DATE,
        _amount_flag   STRING
    )
    USING DELTA
''')

print(f'Table ready: {TABLE}')

## Step 4 — SCD1 MERGE

Update if `transaction_id` already exists, insert if new.
Safe to re-run — identical rows result in a no-op update.

In [ ]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, TABLE)

target.alias('t').merge(
    silver_df.alias('s'),
    't.transaction_id = s.transaction_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print('MERGE complete')

## Step 5 — Verify

In [ ]:
result = spark.table(TABLE)
print(f'Total rows in {TABLE}: {result.count()}')
print(f'Amount mismatches flagged: {result.filter(col("_amount_flag").isNotNull()).count()}')
print('Payment mode distribution:')
result.groupBy('payment_mode').count().display()
result.display()